## 9. CI/CD PIPELINE

Code are following AAI 540 Lab 6
## Setup the load from S3

In [1]:
# If you're using the default bucket, set DEFAULT_BUCKET = True; otherwise, if you're using a specific bucket, set it to False instead
DEFAULT_BUCKET = False

In [2]:
import boto3
import sagemaker
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep, TransformStep, TrainingStep #, ModelStep
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.model_step import ModelStep
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.transformer import Transformer
from sagemaker.model import Model
from sagemaker.workflow.conditions import ConditionLessThanOrEqualTo
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.functions import JsonGet
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.inputs import TrainingInput, TransformInput, CreateModelInput
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.estimator import Estimator

import pandas as pd
import json
from pyathena import connect
sess = sagemaker.Session()
if DEFAULT_BUCKET is True:
    # Code to read/write using the default bucket
    bucket = sess.default_bucket()
else:
    # Code to use a previously existing bucket
    bucket = "usdmsaai540-spring2026-team1"
s3 = boto3.resource("s3")
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
pipeline_session = PipelineSession()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
# Set S3 path to Parquet data and S3 Prefix
s3_path_parquet = f's3://{bucket}/CCPP/data//parquet'
s3_prefix = 'CCPP-enery-prediction-linear-regression'

# Set Athena parameters
database_name = "ccpp_aws_fp"
table_name_csv = "final_data_csv"
table_name_parquet = "final_data_parquet"

# Preparing the input and batch data uris
job_name = 'xg_2026-02-18-07-13-00' # Job name from V1

input_data_uri = f's3://{bucket}/{s3_prefix}/testing/' # Using the Testing data as input
batch_data_uri = f's3://{bucket}/{s3_prefix}/batch/' # Using the batch data as batch


In [4]:
# Preparing the model artifact parameters
model_artifact_param = ParameterString(
    name="ModelArtifactPath",
    default_value=f"s3://{bucket}/{s3_prefix}/{job_name}/{job_name}/output/model.tar.gz"
)

# Preparing the batch data parameters
batch_data_param = ParameterString(
    name="BatchDataPath",
    default_value=f"s3://{bucket}/{s3_prefix}/batch/"
)

from sagemaker.workflow.lambda_step import LambdaStep, LambdaOutput
from sagemaker.workflow.parameters import ParameterString
model = Model(
    image_uri=sagemaker.image_uris.retrieve(
        framework="linear-learner",
        region=region
    ),
    model_data=model_artifact_param,
    role=role
)


model_name_param = ParameterString(
    name="ModelName",
    default_value="existing-model-from-artifact"
)

step_register = LambdaStep(
    name="CreateModelViaLambda",
    lambda_func=lambda_handler,  # your deployed Lambda
    inputs={
        "ModelName": model_name_param,
        "ImageUri": model.image_uri,
        "ModelDataUrl": model_artifact_param,
        "RoleArn": role
    },
    outputs=[
        LambdaOutput(output_name="ModelName", output_type="String")
    ]
)

In [5]:
output_location = "s3://{}/{}/output/{}".format(bucket, s3_prefix, job_name)
output_location

's3://usdmsaai540-spring2026-team1/CCPP-enery-prediction-linear-regression/output/xg_2026-02-18-07-13-00'

## CI/CD Pipeline
### Preparing the model parameters

In [6]:
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")
model_approval_status = ParameterString(
    name="ModelApprovalStatus", default_value="PendingManualApproval"
)
input_data = ParameterString(
    name="InputData",
    default_value=input_data_uri,
)
batch_data = ParameterString(
    name="BatchData",
    default_value=batch_data_uri,
)
mse_threshold = ParameterFloat(name="MseThreshold", default_value=2.721507) #Values obtained during training in research
r2_threshold = ParameterFloat(name="R2Threshold", default_value=0.986821) #Values obtained during training in research

## Defining a Training step

In [7]:
# Preparing the configuration for the model path and training parameters
model_path = f"s3://{bucket}/CCPP/CCPPTrain"
image_uri = sagemaker.image_uris.retrieve(
    framework="xgboost",
    region=region,
    version="1.0-1",
    py_version="py3",
    instance_type="ml.m5.xlarge",
)
xgb_train = Estimator(
    image_uri=image_uri,
    instance_type=instance_type,
    instance_count=1,
    output_path=model_path,
    role=role,
    sagemaker_session=pipeline_session,
)
xgb_train.set_hyperparameters(
    objective="reg:linear",
    num_round=50,
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.7,
)

train_args = xgb_train.fit(
    inputs={
        "train": TrainingInput(
            s3_data=f's3://{bucket}/{s3_prefix}/train/',
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=f's3://{bucket}/{s3_prefix}/validation/',
            content_type="text/csv",
        ),
    }
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [8]:
# Preparing the training step
step_train = TrainingStep(
    name="CCPPTrain",
    step_args=train_args,
)

## Defining the evaluation script and step to evaluate the model

In [9]:
!mkdir -p code

In [10]:
%%writefile code/evaluation.py
import json
import pathlib
import pickle
import tarfile

import joblib
import numpy as np
import pandas as pd
import xgboost

from sklearn.metrics import mean_squared_error, r2_score


if __name__ == "__main__":
    model_path = f"/opt/ml/processing/model/model.tar.gz"
    with tarfile.open(model_path) as tar:
        tar.extractall(path=".")

    #model = pickle.load(open("xgboost-model", "rb"))
    #model = pickle.load(open("model_algo-1", "rb"))
    model = xgboost.Booster()
    #model.load_model("model_algo-1")
    model.load_model("xgboost-model")
    

    test_path = "/opt/ml/processing/test/test_data.csv"
    df = pd.read_csv(test_path, header=None)

    y_test = df.iloc[:, -1].to_numpy()
    df.drop(df.columns[-1], axis=1, inplace=True)

    X_test = xgboost.DMatrix(df.values)

    predictions = model.predict(X_test)

    mse = mean_squared_error(y_test, predictions)
    std = np.std(y_test - predictions)
    
    r2 = r2_score(y_test, predictions)
    
    report_dict = {
        "regression_metrics": {
            "mse": {
                "value": mse,
                "standard_deviation": std
            },
            "r2": {
                "value": r2
            },
        },
    }
    output_dir = "/opt/ml/processing/evaluation"
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)

    evaluation_path = f"{output_dir}/evaluation.json"
    with open(evaluation_path, "w") as f:
        f.write(json.dumps(report_dict))

Overwriting code/evaluation.py


In [11]:
# Configuring the Evalution script and evalution arguments
script_eval = ScriptProcessor(
    image_uri=image_uri,
    command=["python3"],
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="script-ccpp-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = script_eval.run(
    inputs=[
        ProcessingInput(
            source=f's3://{bucket}/{s3_prefix}/output/{job_name}/{job_name}/output/model.tar.gz',
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=f's3://{bucket}/{s3_prefix}/testing/',
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[
        ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation"),
    ],
    code="code/evaluation.py",
)

In [12]:
# Configuring an evaluation report to store the results of the evaluation step
evaluation_report = PropertyFile(
    name="EvaluationReport", output_name="evaluation", path="evaluation.json"
)

# Configuring the evaluation step
step_eval = ProcessingStep(
    name="CCPPEval",
    step_args=eval_args,
    property_files=[evaluation_report],
)

In [13]:
# Defining the model from the existing model.tar.gz file
model = Model(
    image_uri=sagemaker.image_uris.retrieve(
        framework="linear-learner",
        region=region
    ),
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    sagemaker_session=pipeline_session,
    role=role
)

INFO:sagemaker.image_uris:Same images used for training and inference. Defaulting to image scope: inference.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


## CONDITIONAL STEP
### "IF" Portion
#### It includes 3 sub-steps: 
- Model Creation
- Batch Transform
- Model Register

In [14]:
# Configuring the create model step
step_create_model = ModelStep(
    name="CCPPCreateModel",
    step_args=model.create(instance_type="ml.m5.large", accelerator_type="ml.eia1.medium"),
)

In [15]:
# Configuring the Transformer for inference
transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    output_path=f"s3://{bucket}/{s3_prefix}/CCPPTransform",
)

In [16]:
# Defining the Transform Step
step_transform = TransformStep(
    name="CCPPTransform", transformer=transformer, inputs=TransformInput(data=batch_data)
)

In [17]:
model_package_group_name = f"CCPPModelPackageGroupName"
# Defining the metrics to evaluate
model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri="{}/evaluation.json".format(
            step_eval.arguments["ProcessingOutputConfig"]["Outputs"][0]["S3Output"]["S3Uri"]
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/csv"],
    response_types=["text/csv"],
    inference_instances=["ml.t2.medium", "ml.m5.xlarge"],
    transform_instances=["ml.m5.xlarge"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)
step_register = ModelStep(name="CCPPRegisterModel", step_args=register_args)

### "ELSE" Portion
#### This contains only the "Fail" step, which will be executed when the new model does not meet the criteria

In [18]:
# Defining the Fail Step
step_fail = FailStep(
    name="CCPPMSEAndR2Fail",
    error_message=Join(
        on=" ",
        values=[
            "Execution failed due to:",
            # MSE actual value
            "MSE value:",
            JsonGet(
                step_name=step_eval.name,
                property_file=evaluation_report,
                json_path="regression_metrics.mse.value",
            ),
            "exceeded threshold:",
            mse_threshold,

            "and/or",

            # R2 actual value
            "R2 value:",
            JsonGet(
                step_name=step_eval.name,
                property_file=evaluation_report,
                json_path="regression_metrics.r2.value",
            ),
            "fell below threshold:",
            r2_threshold,
        ]
    ),
)

## Defining a condition step to check MSE and R-Squared of the model to decide the pipeline's course of action

In [19]:
# Creating a "less than or equal" condition for the mse
cond_lte_mse = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="regression_metrics.mse.value",
    ),
    right=mse_threshold,
)

# Creating a "less than or equal" condition for the R-squared
cond_lte_r2 = ConditionLessThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="regression_metrics.r2.value",
    ),
    right=r2_threshold,
)
step_cond = ConditionStep(
    name="CCPPMSEAndR2Cond",
    conditions=[cond_lte_mse, cond_lte_r2],
    if_steps=[step_register, step_create_model, step_transform],
    else_steps=[step_fail],
)

## Defining a pipeline using the parameters, steps, and conditions defined above

In [20]:
# Defining the pipeline with all the parameters defined above
pipeline_name = f"CCPPPipeline"
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        instance_type,
        model_approval_status,
        input_data,
        batch_data,
        mse_threshold,
        r2_threshold,
    ],
    steps=[step_train, step_eval, step_cond],
)

In [21]:
# Confirming the pipeline is well-defined and the parameters and step properties resolved correctly
definition = json.loads(pipeline.definition())
definition

{'Version': '2020-12-01',
 'Metadata': {},
 'Parameters': [{'Name': 'ProcessingInstanceCount',
   'Type': 'Integer',
   'DefaultValue': 1},
  {'Name': 'TrainingInstanceType',
   'Type': 'String',
   'DefaultValue': 'ml.m5.xlarge'},
  {'Name': 'ModelApprovalStatus',
   'Type': 'String',
   'DefaultValue': 'PendingManualApproval'},
  {'Name': 'InputData',
   'Type': 'String',
   'DefaultValue': 's3://usdmsaai540-spring2026-team1/CCPP-enery-prediction-linear-regression/testing/'},
  {'Name': 'BatchData',
   'Type': 'String',
   'DefaultValue': 's3://usdmsaai540-spring2026-team1/CCPP-enery-prediction-linear-regression/batch/'},
  {'Name': 'MseThreshold', 'Type': 'Float', 'DefaultValue': 2.721507},
  {'Name': 'R2Threshold', 'Type': 'Float', 'DefaultValue': 0.986821}],
 'PipelineExperimentConfig': {'ExperimentName': {'Get': 'Execution.PipelineName'},
  'TrialName': {'Get': 'Execution.PipelineExecutionId'}},
 'Steps': [{'Name': 'CCPPTrain',
   'Type': 'Training',
   'Arguments': {'AlgorithmSp

## Submitting the pipeline to SageMaker and starting execution

In [22]:
# Submitting the pipeline definition to the Pipeline service, using the session role to create all the jobs defined in the steps
pipeline.upsert(role_arn=role)

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:987390971271:pipeline/CCPPPipeline',
 'ResponseMetadata': {'RequestId': '0115b303-e5f6-4a99-a319-2bee1af92803',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '0115b303-e5f6-4a99-a319-2bee1af92803',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '103',
   'date': 'Sun, 22 Feb 2026 20:23:02 GMT'},
  'RetryAttempts': 0}}

In [23]:
# Starting the pipeline and accepting all the default parameters
execution = pipeline.start()

In [24]:
# Describing the pipeline execution
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:987390971271:pipeline/CCPPPipeline',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:987390971271:pipeline/CCPPPipeline/execution/9o8piaf2cv98',
 'PipelineExecutionDisplayName': 'execution-1771791782131',
 'PipelineExecutionStatus': 'Executing',
 'CreationTime': datetime.datetime(2026, 2, 22, 20, 23, 2, 80000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 2, 22, 20, 23, 2, 80000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:987390971271:user-profile/d-us8baws1boci/default-1770785083214',
  'UserProfileName': 'default-1770785083214',
  'DomainId': 'd-us8baws1boci',
  'IamIdentity': {'Arn': 'arn:aws:sts::987390971271:assumed-role/LabRole/SageMaker',
   'PrincipalId': 'AROA6LZIWRWD4A4MFA6UD:SageMaker'}},
 'LastModifiedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:987390971271:user-profile/d-us8baws1boci/default-1770785083214',
  'UserProfileName': 'default-1770785083214',
  'Dom

In [25]:
# Waiting for the execution to complete
try:
    execution.wait()
except Exception as error:
    print(error)

Waiter PipelineExecutionComplete failed: Waiter encountered a terminal failure state: For expression "PipelineExecutionStatus" we matched expected path: "Failed"


In [26]:
# Listing the steps in the execution or confirmation
execution.list_steps()

[{'StepName': 'CCPPMSEAndR2Fail',
  'StartTime': datetime.datetime(2026, 2, 22, 20, 25, 38, 440000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 2, 22, 20, 25, 38, 726000, tzinfo=tzlocal()),
  'StepStatus': 'Failed',
  'FailureReason': 'Execution failed due to: MSE value: 3.90598671066806 exceeded threshold: 2.721507 and/or R2 value: 0.8994239605182245 fell below threshold: 0.986821',
  'Metadata': {'Fail': {'ErrorMessage': 'Execution failed due to: MSE value: 3.90598671066806 exceeded threshold: 2.721507 and/or R2 value: 0.8994239605182245 fell below threshold: 0.986821'}},
  'AttemptCount': 1},
 {'StepName': 'CCPPMSEAndR2Cond',
  'StartTime': datetime.datetime(2026, 2, 22, 20, 25, 37, 964000, tzinfo=tzlocal()),
  'EndTime': datetime.datetime(2026, 2, 22, 20, 25, 38, 196000, tzinfo=tzlocal()),
  'StepStatus': 'Succeeded',
  'Metadata': {'Condition': {'Outcome': 'False'}},
  'AttemptCount': 1},
 {'StepName': 'CCPPTrain',
  'StartTime': datetime.datetime(2026, 2, 22, 20, 23, 

## Kernel Shutdown

In [27]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>